# CALM — 收敛时间 & Outer-Iter 数 vs d

固定 n=1000, degree=2，对 d∈{50,60,70,80} 测量 CALM 增广拉格朗日外循环收敛所需的迭代数和时间。

收敛判据：h_acyclicity ≤ htol（主）或连续 CONV_PATIENCE 个外循环分数基本不变（辅）。

注：SUBPROBLEM_ITER 设为较小值以使实验可行；增大可得更精确解但更慢。

In [ ]:
import os, sys, time, warnings
import concurrent.futures
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "calm_dataset.py").exists() and (p / "coordinate_descent").exists():
            return p
    raise RuntimeError(f"repo root not found from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from calm_dataset import CalmDataset
print(f"REPO_ROOT: {REPO_ROOT}")

In [ ]:
ALGO_NAME = "CALM"
TAG       = "calm_convergence"

sys.path.insert(0, str(REPO_ROOT / "external" / "CALM"))
import torch
from CALMUtils import CalmModel
from iamb import iamb_markov_network
print("CALM modules: OK")

In [ ]:
D_LIST          = [50, 60, 70, 80]
N_SAMPLES       = 1000
TRIALS          = 3
DEGREE          = 2.0
GRAPH_TYPE      = "ER"
SEED_BASE       = 42
TIMEOUT_SEC     = 3600

SUBPROBLEM_ITER = 500    # inner Adam steps per outer AL iteration
T_MAX_OUTER     = 200    # max outer AL iterations
HTOL            = 1e-8   # acyclicity convergence threshold
CONV_TOL        = 1e-4   # relative loss change threshold (secondary criterion)
CONV_PATIENCE   = 5      # consecutive non-improving outer iters before stopping

os.makedirs(str(REPO_ROOT / "experiments" / "results"), exist_ok=True)

In [ ]:
def run_with_result_and_timeout(fn, timeout):
    """Returns (result, elapsed_sec, error_str_or_None)."""
    holder = [None]
    def _wrap():
        holder[0] = fn()
    ex = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    fut = ex.submit(_wrap)
    t0 = time.perf_counter()
    try:
        fut.result(timeout=timeout)
        elapsed = time.perf_counter() - t0
        ex.shutdown(wait=False)
        return holder[0], elapsed, None
    except concurrent.futures.TimeoutError:
        ex.shutdown(wait=False)
        return None, float(timeout), "TIMEOUT"
    except Exception as e:
        elapsed = time.perf_counter() - t0
        ex.shutdown(wait=False)
        return None, elapsed, str(e)


def make_data(d, seed):
    ds = CalmDataset(
        n=N_SAMPLES, d=d, graph_type=GRAPH_TYPE,
        degree=DEGREE, sem_type="gauss",
        noise_ratio=4.0, seed=int(seed),
    )
    return ds.X

In [ ]:
def run_algo(X):
    """Run CALM outer loop with convergence tracking.
    Returns (n_outer_iters, final_h, final_loss, converged_by_h).
    """
    np.random.seed(0)
    torch.manual_seed(0)

    d = X.shape[1]
    X_c = X - np.mean(X, axis=0, keepdims=True)
    cov_emp = np.cov(X_c.T, bias=True)
    moral_mask, _ = iamb_markov_network(X_c, alpha=0.01)

    device = torch.device("cpu")
    cov_t  = torch.from_numpy(cov_emp).double().to(device)
    mask_t = torch.from_numpy(moral_mask).float().to(device)

    model = CalmModel(d, mask_t, tau=0.5, lambda1=0.005).to(device)

    rho = 1e-5
    loss_history = []
    h_history    = []
    no_improve   = 0
    converged_by_h = False

    for outer in range(T_MAX_OUTER):
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        for _ in range(SUBPROBLEM_ITER):
            opt.zero_grad()
            loss = model.compute_loss(cov_t, rho)
            loss.backward(retain_graph=True)
            opt.step()

        with torch.no_grad():
            B_logit_c = model.B_logit.detach().clone()
            B_logit_c[model.moral_mask == 0] = float("-inf")
            h = model.compute_h(torch.sigmoid(B_logit_c / model.tau)).item()
            loss_val = model.compute_loss(cov_t, rho).item()

        h_history.append(h)
        loss_history.append(loss_val)

        # Primary convergence: acyclicity constraint satisfied
        if h <= HTOL:
            converged_by_h = True
            break

        # Secondary convergence: loss plateau
        if len(loss_history) > CONV_PATIENCE:
            ref = loss_history[-(CONV_PATIENCE + 1)]
            rel = abs(ref - loss_val) / max(1.0, abs(ref))
            if rel < CONV_TOL:
                no_improve += 1
                if no_improve >= CONV_PATIENCE:
                    break
            else:
                no_improve = 0

        rho *= 3.0
        if rho > 1e16:
            break

    n_outer = len(h_history)
    return n_outer, h_history[-1], loss_history[-1], converged_by_h

## 主循环

In [ ]:
records = []
rng = np.random.default_rng(SEED_BASE)
seeds = rng.integers(0, 10**9, size=(len(D_LIST), TRIALS))

for di, d in enumerate(D_LIST):
    for t in range(TRIALS):
        X = make_data(d, int(seeds[di, t]))
        result, elapsed, err = run_with_result_and_timeout(
            lambda X=X: run_algo(X), TIMEOUT_SEC
        )
        if err == "TIMEOUT":
            print(f"d={d:3d}  trial={t+1}  TIMEOUT (>{TIMEOUT_SEC}s)")
            records.append({"d": d, "trial": t+1, "n_outer_iters": np.nan,
                            "time_sec": elapsed, "final_h": np.nan,
                            "final_loss": np.nan, "converged_by_h": False,
                            "converged": False})
        elif err:
            print(f"d={d:3d}  trial={t+1}  ERROR: {err}")
        else:
            n_outer, final_h, final_loss, conv_h = result
            conv_tag = "h-conv" if conv_h else "loss-plateau"
            print(f"d={d:3d}  trial={t+1}  outer_iters={n_outer:3d}  "
                  f"time={elapsed:7.2f}s  h={final_h:.3e}  [{conv_tag}]")
            records.append({"d": d, "trial": t+1, "n_outer_iters": n_outer,
                            "time_sec": elapsed, "final_h": final_h,
                            "final_loss": final_loss, "converged_by_h": conv_h,
                            "converged": True})

df = pd.DataFrame(records)
print()
print(df.to_string(index=False))

## 汇总 & 可视化

In [ ]:
df_conv = df[df["converged"]]
agg = df_conv.groupby("d").agg(
    mean_iters=("n_outer_iters", "mean"),
    std_iters=("n_outer_iters", "std"),
    mean_time=("time_sec", "mean"),
    std_time=("time_sec", "std"),
    mean_h=("final_h", "mean"),
    pct_h_conv=("converged_by_h", "mean"),
).reset_index()
print(agg.to_string(index=False))

if len(agg) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f"{ALGO_NAME} — convergence vs d  (n={N_SAMPLES}, ER degree={DEGREE}, "
                 f"subproblem_iter={SUBPROBLEM_ITER})", fontsize=12)

    ax = axes[0]
    ax.errorbar(agg["d"], agg["mean_iters"], yerr=agg["std_iters"].fillna(0),
                fmt="o-", capsize=4)
    ax.set_xlabel("d"); ax.set_ylabel("outer AL iters to converge")
    ax.set_title("Outer iterations")
    ax.grid(True, ls="--", alpha=0.4)

    ax = axes[1]
    ax.errorbar(agg["d"], agg["mean_time"], yerr=agg["std_time"].fillna(0),
                fmt="o-", capsize=4, color="orange")
    ax.set_xlabel("d"); ax.set_ylabel("time (s)")
    ax.set_title("Time to convergence")
    ax.grid(True, ls="--", alpha=0.4)

    ax = axes[2]
    ax.semilogy(agg["d"], agg["mean_h"], "o-", color="green")
    ax.axhline(HTOL, ls="--", color="red", label=f"htol={HTOL}")
    ax.set_xlabel("d"); ax.set_ylabel("final h (log scale)")
    ax.set_title("Acyclicity h at convergence")
    ax.legend(); ax.grid(True, ls="--", alpha=0.4)

    plt.tight_layout()
    out_png = REPO_ROOT / "experiments" / "results" / f"{TAG}.png"
    plt.savefig(out_png, dpi=120)
    plt.show()
    print(f"figure saved → {out_png}")
else:
    print("Not enough data to plot.")

## 结论

In [ ]:
n_total = len(df)
n_conv  = df["converged"].sum()
n_h_conv = df["converged_by_h"].sum()
print(f"Converged: {n_conv}/{n_total} trials")
print(f"  h-convergence (h ≤ {HTOL}): {n_h_conv}/{n_total}")
print(f"  loss-plateau fallback: {n_conv - n_h_conv}/{n_total}")
print(f"Config: SUBPROBLEM_ITER={SUBPROBLEM_ITER}, T_MAX_OUTER={T_MAX_OUTER}, "
      f"CONV_TOL={CONV_TOL}, CONV_PATIENCE={CONV_PATIENCE}")